In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [9]:
def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def solvenominal (sets,p,R,r,m,r_f,c,ordering):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

In [11]:
def cut_plane(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 1
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    obj_value = prob.value
    [rbvalue,q_b] = robustcheck(w,R,r,p,m,r_f)
    nonstop = True
    while nonstop:
        print(rbvalue)
        if rbvalue <= c+1e-5:
            return(w,obj_value,iterations)
        h = q_b
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
        obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        obj_value = prob.value
        w = a.value
        [rbvalue,q_b] = robustcheck(w,R,r,p,m,r_f)
        iterations = iterations + 1
        print(iterations)
    

In [4]:
np.random.seed(5)

In [13]:
N=30
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))

[0.05932743 0.04215153 0.0560011 ]


In [6]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.12

In [14]:
cut_plane(R,r,c,p,m,r_f)

7.151405400540973
2
3.4454892681071896
3
0.30891423174692634
4
0.1372272046050345
5
0.15299871682426536
6
0.12000999771031436


(array([ 0.42786093, -0.08469687,  0.46051803]), 0.04779962170921299, 6)

In [14]:
N=100
p = np.zeros(N)+1/N
I = 20
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))

[0.04407127 0.10951127 0.04444003 0.03499775 0.03166994 0.02809319
 0.04240501 0.0532941  0.0670439  0.0357301  0.04737415 0.06765955
 0.00438214 0.05828868 0.03276304 0.05454197 0.04507403 0.04631197
 0.06829721 0.07897851]


In [16]:
r = 0.3
m = 0.95    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.01
print(cut_plane(R,r,c,p,m,r_f))

0.3833893960148599
2
0.372753014207287
3
0.19513597211521572
4
0.2516640457909745
5
0.20536870510658028
6
0.15059004725460642
7
0.16638091000569383
8
0.07402462974990108
9
0.08046068529088346
10
0.06714698500928318
11
0.0757973468084087
12
0.1206522297354348
13
0.12145920758919639
14
0.0616887272860344
15
0.05533348993943844
16
0.07368981460053256
17
0.027138455969631263
18
0.04568447315371218
19
0.03535307655457256
20
0.02198524769504428
21
0.02766816230907562
22
0.010009966022417893
(array([ 5.65137719e-02,  6.44771026e-02,  2.27717088e-02,  1.93038487e-02,
        6.27355084e-03,  4.39017202e-02,  1.55434150e-12,  1.37786453e-01,
       -1.16013390e-13,  8.49210134e-11,  6.93237549e-03,  6.48984686e-02,
        3.28921243e-02,  1.28608749e-01,  6.37075534e-02,  6.97254848e-04,
        1.04642228e-01,  4.55848952e-02,  8.49320170e-03,  5.67342773e-02]), 0.10951126623539337, 22)


In [23]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
psets

[[0], [1], [2], [0, 1], [0, 2], [1, 2], [0, 1, 2]]

In [34]:
h = np.zeros(N)
    
for i in range(N-1):
    h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
h[N-1]=h_3(p[N-1],m)

for ind in psets:
     print(sum(h[ind])>h_3(sum(p[ind]),m))

False
False
False
False
False
False
False
